# 03 - V2 Raw Keepa Data Exploration

This notebook explores the raw V2 Keepa JSON files before cleaning and processing.

The goal is to understand what information was actually collected from Keepa before deciding how to clean, transform, and extract features for the modelling pipeline.

This notebook answers:

- How many raw JSON files were saved?
- What does one raw product JSON file contain?
- Which fields are available at the top level?
- Which price, sales rank, offer count, and availability-related fields exist?
- Are the raw files structurally consistent across products?
- Are there unrealistic raw values or outliers that need cleaning?
- Which metadata fields should be extracted into the processed modelling dataset?
- What implications do the raw data findings have for price-drop prediction?

The raw data folder is:

`data/raw/v2_raw/`

A summary and interpretation of the main findings is provided at the end of the notebook. This final section explains how the raw data exploration informs the cleaning strategy, feature extraction choices, and later XGBoost modelling decisions.

In [5]:
# Import Libraries and Set Display Options

import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.colheader_justify", "left")

In [6]:
# Locate Raw V2 Keepa JSON Files

RAW_DIR = Path("../data/raw/v2_raw")

raw_files = sorted(RAW_DIR.glob("*_raw.json"))

print("Raw data directory exists:", RAW_DIR.exists())
print("Number of raw JSON files found:", len(raw_files))

if len(raw_files) > 0:
    print("First raw file:", raw_files[0].name)
    print("Last raw file:", raw_files[-1].name)
else:
    print("No raw JSON files found. Check the RAW_DIR path.")

Raw data directory exists: True
Number of raw JSON files found: 966
First raw file: B00002N6J9_raw.json
Last raw file: B09PGSZDHH_raw.json


In [7]:
# Load and Inspect One Raw Product JSON File

example_file = raw_files[0]

with open(example_file, "r", encoding="utf-8") as f:
    example_product = json.load(f)

top_level_fields = sorted(example_product.keys())

print("Example file:", example_file.name)
print("Object type:", type(example_product))
print("Number of top-level fields:", len(top_level_fields))

top_level_fields_df = pd.DataFrame({"top_level_field": top_level_fields})

with pd.option_context("display.max_rows", 100):
    display(top_level_fields_df)

Example file: B00002N6J9_raw.json
Object type: <class 'dict'>
Number of top-level fields: 92


,top_level_field
0,asin
1,author
2,availabilityAmazon
3,availabilityAmazonDelay
4,binding
5,brand
6,buyBoxEligibleOfferCounts
7,buyBoxSellerIdHistory
8,categories
9,categoryTree


In [9]:
# Inspect Important Raw JSON Fields

important_fields = [
    "asin",
    "title",
    "brand",
    "categoryTree",
    "categories",
    "rootCategory",
    "csv",
    "data",
    "salesRanks",
    "buyBoxEligibleOfferCounts",
    "availabilityAmazon",
    "monthlySold",
    "monthlySoldHistory",
    "hasReviews",
    "lastRatingUpdate",
    "lastPriceChange",
    "trackingSince"
]

field_summary = []

for field in important_fields:
    value = example_product.get(field, None)
    
    if isinstance(value, dict):
        preview = list(value.keys())[:10]
        length = len(value)
    elif isinstance(value, list):
        preview = value[:3]
        length = len(value)
    else:
        preview = value
        length = None
    
    field_summary.append({
        "field": field,
        "type": type(value).__name__,
        "length_if_list_or_dict": length,
        "preview": preview
    })

field_summary_df = pd.DataFrame(field_summary)
field_summary_df

,field,type,length_if_list_or_dict,preview
0,asin,str,NaN,B00002N6J9
1,title,str,NaN,"Filtrete 16x20x1 Air Filter, MERV 11, MPR 1000, Allergens & Pet Dander, 3-Month AC and Furnace Filters, Cleaner Air, Reliable Airflow, 4 HVAC Filt..."
2,brand,str,NaN,Filtrete
3,categoryTree,list,5.0,"[{'catId': 228013, 'name': 'Tools & Home Improvement'}, {'catId': 551240, 'name': 'Building Supplies'}, {'catId': 495346, 'name': 'HVAC'}]"
4,categories,list,1.0,[13399891]
5,rootCategory,int,NaN,228013
6,csv,list,36.0,"[[5361728, 4650, 5364522, 4711, 5366064, 4659, 5367496, 4678, 5371132, 4601, 5372188, 4605, 5372596, -1, 5399740, 4633, 5404712, 4692, 5407712, 46..."
7,data,dict,24.0,"[AMAZON_time, AMAZON, df_AMAZON, NEW_time, NEW, df_NEW, USED_time, USED, df_USED, SALES_time]"
8,salesRanks,dict,4.0,"[228013, 3741191, 13397451, 13399891]"
9,buyBoxEligibleOfferCounts,list,8.0,"[2, 8, 0]"


In [10]:
# Inspect Named Time-Series Fields Inside the Raw Data Object

data_object = example_product.get("data", {})

print("Type of data object:", type(data_object))
print("Number of fields inside data object:", len(data_object))

data_fields = sorted(data_object.keys())

data_fields_df = pd.DataFrame({"data_field": data_fields})

data_fields_df.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

Type of data object: <class 'dict'>
Number of fields inside data object: 24


,data_field
0,AMAZON
1,AMAZON_time
2,COUNT_NEW
3,COUNT_NEW_time
4,COUNT_REFURBISHED
5,COUNT_REFURBISHED_time
6,COUNT_USED
7,COUNT_USED_time
8,LISTPRICE
9,LISTPRICE_time


In [11]:
# Check Availability of Important Data Fields Across Raw Files

important_data_fields = [
    "AMAZON",
    "NEW",
    "SALES",
    "COUNT_NEW",
    "LISTPRICE",
    "USED",
    "COUNT_USED",
    "COUNT_REFURBISHED"
]

field_availability = []

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    data_obj = product.get("data", {})
    asin = product.get("asin", file.stem.replace("_raw", ""))
    
    row = {"asin": asin}
    
    for field in important_data_fields:
        values = data_obj.get(field, None)
        
        if isinstance(values, list):
            row[f"{field}_exists"] = True
            row[f"{field}_length"] = len(values)
            row[f"{field}_non_empty"] = len(values) > 0
        else:
            row[f"{field}_exists"] = False
            row[f"{field}_length"] = 0
            row[f"{field}_non_empty"] = False
    
    field_availability.append(row)

field_availability_df = pd.DataFrame(field_availability)

summary_rows = []

for field in important_data_fields:
    summary_rows.append({
        "field": field,
        "files_with_field": field_availability_df[f"{field}_exists"].sum(),
        "files_with_non_empty_field": field_availability_df[f"{field}_non_empty"].sum(),
        "average_length": field_availability_df[f"{field}_length"].mean(),
        "median_length": field_availability_df[f"{field}_length"].median(),
        "max_length": field_availability_df[f"{field}_length"].max()
    })

field_availability_summary = pd.DataFrame(summary_rows)

field_availability_summary.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,field,files_with_field,files_with_non_empty_field,average_length,median_length,max_length
0,AMAZON,966,966,1124.096273,642.000000,13624
1,NEW,966,966,1432.953416,873.000000,12121
2,SALES,966,966,13947.563147,12064.000000,41128
3,COUNT_NEW,966,966,2463.375776,1425.500000,17627
4,LISTPRICE,966,966,280.873706,185.000000,3210
5,USED,966,966,1041.379917,743.500000,7993
6,COUNT_USED,946,946,1530.324017,845.000000,16607
7,COUNT_REFURBISHED,797,797,77.179089,1.000000,7813


In [12]:
# Search Raw JSON Keys for Rating and Review Information

search_terms = ["rating", "review"]

matched_keys = Counter()

def search_keys(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            full_key = f"{prefix}.{key}" if prefix else key
            
            if any(term in key.lower() for term in search_terms):
                matched_keys[full_key] += 1
            
            search_keys(value, full_key)
    
    elif isinstance(obj, list):
        for item in obj[:5]:
            search_keys(item, prefix)

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    search_keys(product)

matched_keys_df = pd.DataFrame(
    matched_keys.items(),
    columns=["matched_key", "count"]
).sort_values("count", ascending=False)

matched_keys_df.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,matched_key,count
0,hasReviews,966
1,lastRatingUpdate,966
2,stats_parsed.isLowest.RATING,50
3,stats_parsed.isLowest.COUNT_REVIEWS,50
4,stats_parsed.isLowest90.RATING,50
5,stats_parsed.isLowest90.COUNT_REVIEWS,50


In [13]:
# Check Availability of Useful Product Metadata Across Raw Files

metadata_fields = [
    "asin",
    "title",
    "brand",
    "manufacturer",
    "categoryTree",
    "rootCategory",
    "productGroup",
    "monthlySold",
    "monthlySoldHistory",
    "availabilityAmazon",
    "trackingSince",
    "listedSince",
    "lastPriceChange"
]

metadata_rows = []

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    row = {}
    
    for field in metadata_fields:
        value = product.get(field, None)
        
        if isinstance(value, list):
            row[field] = len(value) if len(value) > 0 else np.nan
        elif isinstance(value, dict):
            row[field] = len(value) if len(value) > 0 else np.nan
        else:
            row[field] = value
    
    metadata_rows.append(row)

metadata_df = pd.DataFrame(metadata_rows)

metadata_summary = pd.DataFrame({
    "non_missing_count": metadata_df.notna().sum(),
    "missing_count": metadata_df.isna().sum(),
    "missing_percentage": metadata_df.isna().mean() * 100
}).sort_values("missing_percentage")

metadata_summary.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,non_missing_count,missing_count,missing_percentage
asin,966,0,0.000000
title,966,0,0.000000
brand,966,0,0.000000
categoryTree,966,0,0.000000
rootCategory,966,0,0.000000
monthlySoldHistory,966,0,0.000000
availabilityAmazon,966,0,0.000000
trackingSince,966,0,0.000000
listedSince,966,0,0.000000
lastPriceChange,966,0,0.000000


In [ ]:
# Preview Useful Product Metadata Values

metadata_preview = metadata_df[
    [
        "asin",
        "title",
        "brand",
        "manufacturer",
        "categoryTree",
        "rootCategory",
        "monthlySold",
        "monthlySoldHistory",
        "availabilityAmazon",
        "trackingSince",
        "listedSince",
        "lastPriceChange"
    ]
].head(10)

metadata_preview.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,asin,title,brand,manufacturer,categoryTree,rootCategory,monthlySold,monthlySoldHistory,availabilityAmazon,trackingSince,listedSince,lastPriceChange
0,B00002N6J9,"Filtrete 16x20x1 Air Filter, MERV 11, MPR 1000, Allergens & Pet Dander, 3-Month AC and Furnace Filters, Cleaner Air, Reliable Airflow, 4 HVAC Filters (Actual Size 15.68 x 19.68 x 0.81 in)",Filtrete,3M,5,228013,50.000000,22,0,5353108,321600,8104462
1,B00002N86A,"Kidde Carbon Monoxide Detector, Plug-In with 9V Battery Backup, Digital Display, 85 dB Alarm, LED Status Light Indicators, 3rd Edition",Kidde,Kidde,4,228013,500.000000,760,0,1688280,-2082240,8107536
2,B00004RB5H,"Kotap 20-ft x 30-ft General Purpose Blue Poly Tarp, Item: TRA-2030",Kotap,Kotap America Ltd.,4,228013,50.000000,74,0,141360,-2103840,8100428
3,B00004SY4H,"Sennheiser HD 600 - Audiophile Open-Back Dynamic Wired Headphones Over Ear with Natural Soundstage and Premium Comfort for Music Lovers, Open Metal Earpiece Covers, Black",Sennheiser,Sennheiser,4,172282,300.000000,828,0,1807740,-5590080,8107624
4,B0000510R4,"Tripp Lite Isobar ISOBLOK2-0 Heavy Duty Outlet Extender, Wall Power Strip Surge Protector, 2 Outlets, Direct Plug-in, White, Metal Power Strip, Industrial Garage Work Shop Bench",TRIPP LITE,Tripp Lite,4,172282,200.000000,358,0,119040,-2559916,8105404
5,B0000511U7,"Tripp Lite ISOBAR8ULTRA Isobar 8 Outlet Heavy Duty Power Strip Surge Protector, 3840 Joules, 12ft Cord, Flat Plug, Metal Power Strip, Industrial Garage Work Shop Bench, Under Desk and Wall Mountable",TRIPP LITE,Tripp Lite,4,172282,400.000000,238,0,119040,4807380,8106792
6,B000051ZHS,Intex Explorer 200 Inflatable 2 Person River Boat Raft Set with 2 Oars & Pump,INTEX,Intex,4,3375251,200.000000,444,-1,2136900,-2035880,8107212
7,B000058AKA,"Farberware Classic Series Stainless Steel Stockpot with Lid, 6-Quart Pot",Farberware,Meyer Corporation,5,1055398,100.000000,412,0,119040,-1020192,8105352
8,B000058TJ3,"INTEX 58484EP Swim Center Inflatable Family Pool: 277 Gallon Capacity – 120"" x 72"" x 22"" – Blue",Intex,Intex,4,165793011,400.000000,440,0,2334282,-3463200,8107426
9,B00005NKAA,"Oneida Silverware Set, Satin Sand Dune 20-Piece Everyday Flatware Set, Service For 4, Sand Blasted Finish, 18/0 Stainless Steel, Dishwasher Safe, Knives Spoons And Forks Sets (Silver, 20 Piece)",Oneida,Oneida,4,1055398,50.000000,140,0,3425592,-4860000,8107942


In [ ]:
# Inspect Category Tree Structure and Extract Category Names

category_examples = []

for file in raw_files[:10]:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    category_tree = product.get("categoryTree", [])
    
    category_names = []
    category_ids = []
    
    if isinstance(category_tree, list):
        for item in category_tree:
            if isinstance(item, dict):
                category_names.append(item.get("name"))
                category_ids.append(item.get("catId"))
    
    category_examples.append({
        "asin": product.get("asin"),
        "title": product.get("title"),
        "rootCategory": product.get("rootCategory"),
        "category_names": " > ".join([str(x) for x in category_names if x is not None]),
        "category_ids": category_ids
    })

category_examples_df = pd.DataFrame(category_examples)

category_examples_df.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,asin,title,rootCategory,category_names,category_ids
0,B00002N6J9,"Filtrete 16x20x1 Air Filter, MERV 11, MPR 1000, Allergens & Pet Dander, 3-Month AC and Furnace Filters, Cleaner Air, Reliable Airflow, 4 HVAC Filters (Actual Size 15.68 x 19.68 x 0.81 in)",228013,Tools & Home Improvement > Building Supplies > HVAC > Furnace Parts & Accessories > Furnace Filters,"[228013, 551240, 495346, 322545011, 13399891]"
1,B00002N86A,"Kidde Carbon Monoxide Detector, Plug-In with 9V Battery Backup, Digital Display, 85 dB Alarm, LED Status Light Indicators, 3rd Edition",228013,Tools & Home Improvement > Safety & Security > Fire Safety > Carbon Monoxide Detectors,"[228013, 3180231, 486547011, 495272]"
2,B00004RB5H,"Kotap 20-ft x 30-ft General Purpose Blue Poly Tarp, Item: TRA-2030",228013,Tools & Home Improvement > Hardware > Tarps & Tie-Downs > Tarps,"[228013, 511228, 511388, 511396]"
3,B00004SY4H,"Sennheiser HD 600 - Audiophile Open-Back Dynamic Wired Headphones Over Ear with Natural Soundstage and Premium Comfort for Music Lovers, Open Metal Earpiece Covers, Black",172282,"Electronics > Headphones, Earbuds & Accessories > Headphones & Earbuds > Over-Ear Headphones","[172282, 24046923011, 172541, 12097479011]"
4,B0000510R4,"Tripp Lite Isobar ISOBLOK2-0 Heavy Duty Outlet Extender, Wall Power Strip Surge Protector, 2 Outlets, Direct Plug-in, White, Metal Power Strip, Industrial Garage Work Shop Bench",172282,Electronics > Accessories & Supplies > Power Strips & Surge Protectors > Power Strips,"[172282, 281407, 17854127011, 10967801]"
5,B0000511U7,"Tripp Lite ISOBAR8ULTRA Isobar 8 Outlet Heavy Duty Power Strip Surge Protector, 3840 Joules, 12ft Cord, Flat Plug, Metal Power Strip, Industrial Garage Work Shop Bench, Under Desk and Wall Mountable",172282,Electronics > Accessories & Supplies > Power Strips & Surge Protectors > Power Strips,"[172282, 281407, 17854127011, 10967801]"
6,B000051ZHS,Intex Explorer 200 Inflatable 2 Person River Boat Raft Set with 2 Oars & Pump,3375251,Sports & Outdoors > Sports > Boating & Sailing > Boating,"[3375251, 10971181011, 3421331, 3397331]"
7,B000058AKA,"Farberware Classic Series Stainless Steel Stockpot with Lid, 6-Quart Pot",1055398,Home & Kitchen > Kitchen & Dining > Cookware > Pots & Pans > Stockpots,"[1055398, 284507, 289814, 19343857011, 289832]"
8,B000058TJ3,"INTEX 58484EP Swim Center Inflatable Family Pool: 277 Gallon Capacity – 120"" x 72"" x 22"" – Blue",165793011,Toys & Games > Sports & Outdoor Play > Pools & Water Toys > Kiddie Pools,"[165793011, 166420011, 166437011, 166438011]"
9,B00005NKAA,"Oneida Silverware Set, Satin Sand Dune 20-Piece Everyday Flatware Set, Service For 4, Sand Blasted Finish, 18/0 Stainless Steel, Dishwasher Safe, Knives Spoons And Forks Sets (Silver, 20 Piece)",1055398,Home & Kitchen > Kitchen & Dining > Dining & Entertaining > Flatware,"[1055398, 284507, 13162311, 13218891]"


In [ ]:
# Summarise Root and Leaf Category Coverage Across Raw Files

category_rows = []

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    category_tree = product.get("categoryTree", [])
    
    category_names = []
    category_ids = []
    
    if isinstance(category_tree, list):
        for item in category_tree:
            if isinstance(item, dict):
                category_names.append(item.get("name"))
                category_ids.append(item.get("catId"))
    
    category_rows.append({
        "asin": product.get("asin"),
        "root_category_id": category_ids[0] if len(category_ids) > 0 else np.nan,
        "root_category_name": category_names[0] if len(category_names) > 0 else np.nan,
        "leaf_category_id": category_ids[-1] if len(category_ids) > 0 else np.nan,
        "leaf_category_name": category_names[-1] if len(category_names) > 0 else np.nan,
        "category_depth": len(category_names)
    })

category_df = pd.DataFrame(category_rows)

print("Unique root categories:", category_df["root_category_name"].nunique())
print("Unique leaf categories:", category_df["leaf_category_name"].nunique())

root_category_counts = category_df["root_category_name"].value_counts().reset_index()
root_category_counts.columns = ["root_category_name", "product_count"]

root_category_counts.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

Unique root categories: 6
Unique leaf categories: 480


,root_category_name,product_count
0,Home & Kitchen,337
1,Sports & Outdoors,192
2,Tools & Home Improvement,171
3,Electronics,161
4,Toys & Games,67
5,Appliances,38


In [ ]:
# Inspect Leaf Category Distribution

leaf_category_counts = category_df["leaf_category_name"].value_counts().reset_index()
leaf_category_counts.columns = ["leaf_category_name", "product_count"]

print("Number of unique leaf categories:", category_df["leaf_category_name"].nunique())
print("Number of leaf categories with only 1 product:", (leaf_category_counts["product_count"] == 1).sum())
print("Number of leaf categories with fewer than 5 products:", (leaf_category_counts["product_count"] < 5).sum())

leaf_category_counts.head(20).style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

Number of unique leaf categories: 480
Number of leaf categories with only 1 product: 278
Number of leaf categories with fewer than 5 products: 440


,leaf_category_name,product_count
0,Furnace Filters,20
1,Water Filters,15
2,Sheet & Pillowcase Sets,12
3,Binoculars,11
4,Push Ride-Ons,10
5,Container Sets,9
6,Monitors,9
7,Rice Cookers,9
8,Portable Bluetooth Speakers,8
9,Kick Scooters,8


In [ ]:
# Inspect Time-Series Array Lengths for One Product

example_data = example_product.get("data", {})

time_series_fields = [
    "AMAZON",
    "NEW",
    "SALES",
    "COUNT_NEW",
    "LISTPRICE",
    "USED",
    "COUNT_USED",
    "COUNT_REFURBISHED"
]

time_series_summary = []

for field in time_series_fields:
    values = example_data.get(field, [])
    times = example_data.get(f"{field}_time", [])
    
    time_series_summary.append({
        "field": field,
        "values_length": len(values) if isinstance(values, list) else 0,
        "time_length": len(times) if isinstance(times, list) else 0,
        "first_values": values[:5] if isinstance(values, list) else None,
        "first_times": times[:5] if isinstance(times, list) else None
    })

time_series_summary_df = pd.DataFrame(time_series_summary)

time_series_summary_df.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,field,values_length,time_length,first_values,first_times
0,AMAZON,376,376,"[46.5, 47.11, 46.59, 46.78, 46.01]","['2021-03-12 10:08:00', '2021-03-14 08:42:00', '2021-03-15 10:24:00', '2021-03-16 10:16:00', '2021-03-18 22:52:00']"
1,NEW,1778,1778,"[46.5, 47.11, 46.59, 46.78, 46.01]","['2021-03-12 10:08:00', '2021-03-14 08:42:00', '2021-03-15 10:24:00', '2021-03-16 10:16:00', '2021-03-18 22:52:00']"
2,SALES,6327,6327,"[11, 12, 10, 11, 9]","['2021-03-12 17:24:00', '2021-03-12 23:30:00', '2021-03-13 23:24:00', '2021-03-14 08:42:00', '2021-03-14 23:40:00']"
3,COUNT_NEW,445,445,"[2, 1, -1, 1, 2]","['2021-03-12 10:08:00', '2021-03-19 23:16:00', '2021-04-07 11:00:00', '2021-04-07 12:10:00', '2021-04-07 19:40:00']"
4,LISTPRICE,151,151,"[87.92, nan, 87.92, nan, 87.92]","['2021-03-12 18:40:00', '2021-03-15 17:04:00', '2021-03-22 11:48:00', '2021-04-07 04:42:00', '2021-04-19 15:12:00']"
5,USED,15,15,"[nan, 66.9, 62.69, 58.68, 58.1]","['2021-03-12 10:08:00', '2023-08-26 01:56:00', '2023-08-28 00:16:00', '2023-09-05 05:28:00', '2023-09-08 11:20:00']"
6,COUNT_USED,4,4,"[1, -1, 1, -1]","['2023-08-26 01:56:00', '2023-11-02 01:12:00', '2026-02-14 00:18:00', '2026-02-15 00:10:00']"
7,COUNT_REFURBISHED,1,1,[-1],['2026-02-03 09:56:00']


In [ ]:
# Check Negative and Missing Values in Raw Time-Series Fields

raw_value_quality = []

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    data_obj = product.get("data", {})
    asin = product.get("asin", file.stem.replace("_raw", ""))
    
    for field in time_series_fields:
        values = data_obj.get(field, [])
        
        if isinstance(values, list) and len(values) > 0:
            values_series = pd.Series(values)
            
            raw_value_quality.append({
                "asin": asin,
                "field": field,
                "n_values": len(values_series),
                "n_missing_nan": values_series.isna().sum(),
                "n_negative_one": (values_series == -1).sum(),
                "min_value": values_series.min(skipna=True),
                "max_value": values_series.max(skipna=True)
            })

raw_value_quality_df = pd.DataFrame(raw_value_quality)

raw_value_quality_summary = (
    raw_value_quality_df
    .groupby("field")
    .agg(
        total_values=("n_values", "sum"),
        total_missing_nan=("n_missing_nan", "sum"),
        total_negative_one=("n_negative_one", "sum"),
        min_value=("min_value", "min"),
        max_value=("max_value", "max")
    )
    .reset_index()
)

raw_value_quality_summary["missing_nan_percentage"] = (
    raw_value_quality_summary["total_missing_nan"] / raw_value_quality_summary["total_values"] * 100
)

raw_value_quality_summary["negative_one_percentage"] = (
    raw_value_quality_summary["total_negative_one"] / raw_value_quality_summary["total_values"] * 100
)

raw_value_quality_summary.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,field,total_values,total_missing_nan,total_negative_one,min_value,max_value,missing_nan_percentage,negative_one_percentage
0,AMAZON,1085877,138133,0,5.000000,11474.710000,12.720870,0.000000
1,COUNT_NEW,2379621,0,28251,-1.000000,363.000000,0.000000,1.187206
2,COUNT_REFURBISHED,74555,0,34018,-1.000000,101.000000,0.000000,45.628060
3,COUNT_USED,1478293,0,125837,-1.000000,486.000000,0.000000,8.512318
4,LISTPRICE,271324,83059,0,0.000000,31199.000000,30.612478,0.000000
5,NEW,1384233,75911,0,0.010000,100000.000000,5.483976,0.000000
6,SALES,13473346,0,6081,-1.000000,14245959.000000,0.000000,0.045134
7,USED,1005973,143419,0,0.010000,182520.000000,14.256744,0.000000


In [ ]:
# Inspect Raw Price Outliers for AMAZON and NEW

price_outlier_rows = []

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    asin = product.get("asin", file.stem.replace("_raw", ""))
    title = product.get("title", "")
    brand = product.get("brand", "")
    data_obj = product.get("data", {})
    
    for field in ["AMAZON", "NEW"]:
        values = data_obj.get(field, [])
        times = data_obj.get(f"{field}_time", [])
        
        if isinstance(values, list) and isinstance(times, list):
            for value, time in zip(values, times):
                if pd.notna(value) and (value < 1 or value > 2000):
                    price_outlier_rows.append({
                        "asin": asin,
                        "title": title,
                        "brand": brand,
                        "field": field,
                        "time": time,
                        "value": value
                    })

price_outliers_df = pd.DataFrame(price_outlier_rows)

print("Number of raw AMAZON/NEW price values below $1 or above $2000:", len(price_outliers_df))
print("Number of affected ASINs:", price_outliers_df["asin"].nunique() if len(price_outliers_df) > 0 else 0)

price_outliers_df.head(30).style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

Number of raw AMAZON/NEW price values below $1 or above $2000: 270
Number of affected ASINs: 33


,asin,title,brand,field,time,value
0,B00002N86A,"Kidde Carbon Monoxide Detector, Plug-In with 9V Battery Backup, Digital Display, 85 dB Alarm, LED Status Light Indicators, 3rd Edition",Kidde,NEW,2017-03-20 05:40:00,0.480000
1,B00002N86A,"Kidde Carbon Monoxide Detector, Plug-In with 9V Battery Backup, Digital Display, 85 dB Alarm, LED Status Light Indicators, 3rd Edition",Kidde,NEW,2017-03-20 08:10:00,0.030000
2,B00002N86A,"Kidde Carbon Monoxide Detector, Plug-In with 9V Battery Backup, Digital Display, 85 dB Alarm, LED Status Light Indicators, 3rd Edition",Kidde,NEW,2017-03-22 08:40:00,0.020000
3,B00002N86A,"Kidde Carbon Monoxide Detector, Plug-In with 9V Battery Backup, Digital Display, 85 dB Alarm, LED Status Light Indicators, 3rd Edition",Kidde,NEW,2017-03-22 10:20:00,0.010000
4,B00002N86A,"Kidde Carbon Monoxide Detector, Plug-In with 9V Battery Backup, Digital Display, 85 dB Alarm, LED Status Light Indicators, 3rd Edition",Kidde,NEW,2017-07-07 18:02:00,0.060000
5,B000058TJ3,"INTEX 58484EP Swim Center Inflatable Family Pool: 277 Gallon Capacity – 120"" x 72"" x 22"" – Blue",Intex,NEW,2017-03-09 14:52:00,0.010000
6,B000058TJ3,"INTEX 58484EP Swim Center Inflatable Family Pool: 277 Gallon Capacity – 120"" x 72"" x 22"" – Blue",Intex,NEW,2017-03-13 12:02:00,0.010000
7,B000058TJ3,"INTEX 58484EP Swim Center Inflatable Family Pool: 277 Gallon Capacity – 120"" x 72"" x 22"" – Blue",Intex,NEW,2017-03-15 14:28:00,0.010000
8,B000058TJ3,"INTEX 58484EP Swim Center Inflatable Family Pool: 277 Gallon Capacity – 120"" x 72"" x 22"" – Blue",Intex,NEW,2017-03-16 08:44:00,0.010000
9,B00081NX5U,"Dual Electronics LU43PB 4"" 3-Way High Performance Outdoor Indoor Wired Speakers | Effortless Set Up | Home, Pool, Patio, Garage Use | Weather & UV Resistant | Expansive Stereo Sound Coverage | Black",Dual Electronics,NEW,2015-11-23 09:52:00,0.010000


In [ ]:
# Summarise Raw Price Outliers by Price Field

if len(price_outliers_df) > 0:
    price_outlier_summary = (
        price_outliers_df
        .groupby("field")
        .agg(
            outlier_count=("value", "count"),
            affected_asins=("asin", "nunique"),
            min_outlier_value=("value", "min"),
            max_outlier_value=("value", "max")
        )
        .reset_index()
    )
    
    display(
        price_outlier_summary.style
        .set_properties(**{"text-align": "left"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    )
else:
    print("No raw price outliers found.")

,field,outlier_count,affected_asins,min_outlier_value,max_outlier_value
0,AMAZON,6,2,2677.910000,11474.710000
1,NEW,264,33,0.010000,100000.000000


In [ ]:
# Inspect Monthly Sold and Amazon Availability

monthly_availability_summary = metadata_df[
    ["monthlySold", "availabilityAmazon"]
].describe(include="all").T

display(
    monthly_availability_summary.style
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

print("\nAvailabilityAmazon value counts:")
display(metadata_df["availabilityAmazon"].value_counts(dropna=False).to_frame("count"))

,count,mean,std,min,25%,50%,75%,max
monthlySold,964.000000,1273.599585,3546.060311,50.000000,100.000000,300.000000,1000.000000,60000.000000
availabilityAmazon,966.000000,0.121118,1.113660,-1.000000,0.000000,0.000000,0.000000,4.000000



AvailabilityAmazon value counts:


,count
availabilityAmazon,
0,747
-1,151
4,64
3,4


In [ ]:
# Inspect Monthly Sold History Structure

monthly_history_rows = []

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    history = product.get("monthlySoldHistory", [])
    
    monthly_history_rows.append({
        "asin": product.get("asin"),
        "monthlySold": product.get("monthlySold"),
        "monthlySoldHistory_length": len(history) if isinstance(history, list) else 0,
        "monthlySoldHistory_preview": history[:6] if isinstance(history, list) else None
    })

monthly_history_df = pd.DataFrame(monthly_history_rows)

print("MonthlySoldHistory length summary:")
display(monthly_history_df["monthlySoldHistory_length"].describe())

monthly_history_df.head(10).style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

MonthlySoldHistory length summary:


count     966.000000
mean      438.608696
std       349.588575
min         2.000000
25%       178.000000
50%       377.000000
75%       590.000000
max      2532.000000
Name: monthlySoldHistory_length, dtype: float64

,asin,monthlySold,monthlySoldHistory_length,monthlySoldHistory_preview
0,B00002N6J9,50.000000,22,"[7594328, 50, 7637544, -1, 7644408, 50]"
1,B00002N86A,500.000000,760,"[6732012, 3000, 6746988, 4000, 6752560, 5000]"
2,B00004RB5H,50.000000,74,"[6742148, 50, 6753672, 100, 6754976, 50]"
3,B00004SY4H,300.000000,828,"[6732652, 500, 6763936, 400, 6766992, 300]"
4,B0000510R4,200.000000,358,"[6732076, 400, 6743028, 300, 6744432, 400]"
5,B0000511U7,400.000000,238,"[6732556, 200, 6737868, 300, 6744232, 200]"
6,B000051ZHS,200.000000,444,"[6631780, 3000, 6632402, 2000, 6634472, 10000]"
7,B000058AKA,100.000000,412,"[6631694, 200, 6638170, 100, 6639380, 200]"
8,B000058TJ3,400.000000,440,"[6631692, 300, 6640792, 200, 6656842, 100]"
9,B00005NKAA,50.000000,140,"[6634592, 100, 6638184, 50, 6746606, 100]"


In [ ]:
# Summarise Price History Length per Product

price_history_rows = []

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    data_obj = product.get("data", {})
    
    amazon_values = data_obj.get("AMAZON", [])
    new_values = data_obj.get("NEW", [])
    
    amazon_non_missing = pd.Series(amazon_values).notna().sum() if isinstance(amazon_values, list) else 0
    new_non_missing = pd.Series(new_values).notna().sum() if isinstance(new_values, list) else 0
    
    price_history_rows.append({
        "asin": product.get("asin"),
        "amazon_price_points": len(amazon_values) if isinstance(amazon_values, list) else 0,
        "amazon_non_missing_points": amazon_non_missing,
        "new_price_points": len(new_values) if isinstance(new_values, list) else 0,
        "new_non_missing_points": new_non_missing,
        "best_available_price_points": max(amazon_non_missing, new_non_missing)
    })

price_history_df = pd.DataFrame(price_history_rows)

print("Price history length summary:")
display(price_history_df.describe())

print("\nProducts with fewer than 100 usable AMAZON/NEW price points:")
print((price_history_df["best_available_price_points"] < 100).sum())

price_history_df.sort_values("best_available_price_points").head(20).style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

Price history length summary:


,amazon_price_points,amazon_non_missing_points,new_price_points,new_non_missing_points,best_available_price_points
count,966.000000,966.000000,966.000000,966.00000,966.000000
mean,1124.096273,981.101449,1432.953416,1354.37060,1485.391304
std,1472.598051,1326.624177,1550.424264,1468.06248,1654.947810
min,1.000000,0.000000,50.000000,43.00000,43.000000
25%,306.000000,250.250000,405.500000,360.25000,400.250000
50%,642.000000,535.000000,873.000000,820.50000,899.500000
75%,1355.000000,1156.500000,1912.250000,1824.75000,1933.000000
max,13624.000000,11503.000000,12121.000000,12080.00000,12080.000000



Products with fewer than 100 usable AMAZON/NEW price points:
16


,asin,amazon_price_points,amazon_non_missing_points,new_price_points,new_non_missing_points,best_available_price_points
370,B071G8RWBK,25,14,55,43,43
861,B08TTK3DK2,43,39,53,47,47
157,B00CK018CW,37,21,50,48,48
899,B094PPPR3R,49,48,51,49,49
467,B07CND3JMC,55,44,79,62,62
953,B09K736J9Y,1,0,64,62,62
156,B00CJZ9P6Y,39,20,65,63,63
158,B00CK025GK,55,33,76,70,70
917,B09944VF6S,1,0,106,73,73
692,B082YM66TS,1,0,88,74,74


In [ ]:
# Summarise Sales Rank and Offer Count History Length per Product

rank_offer_history_rows = []

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    data_obj = product.get("data", {})
    
    sales_values = data_obj.get("SALES", [])
    count_new_values = data_obj.get("COUNT_NEW", [])
    
    sales_non_missing = pd.Series(sales_values).notna().sum() if isinstance(sales_values, list) else 0
    count_new_non_missing = pd.Series(count_new_values).notna().sum() if isinstance(count_new_values, list) else 0
    
    rank_offer_history_rows.append({
        "asin": product.get("asin"),
        "sales_rank_points": len(sales_values) if isinstance(sales_values, list) else 0,
        "sales_rank_non_missing_points": sales_non_missing,
        "offer_count_points": len(count_new_values) if isinstance(count_new_values, list) else 0,
        "offer_count_non_missing_points": count_new_non_missing
    })

rank_offer_history_df = pd.DataFrame(rank_offer_history_rows)

print("Sales rank and offer count history length summary:")
display(rank_offer_history_df.describe())

print("\nProducts with fewer than 100 sales rank points:")
print((rank_offer_history_df["sales_rank_non_missing_points"] < 100).sum())

print("\nProducts with fewer than 100 offer count points:")
print((rank_offer_history_df["offer_count_non_missing_points"] < 100).sum())

rank_offer_history_df.sort_values("offer_count_non_missing_points").head(20).style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

Sales rank and offer count history length summary:


,sales_rank_points,sales_rank_non_missing_points,offer_count_points,offer_count_non_missing_points
count,966.000000,966.000000,966.000000,966.000000
mean,13947.563147,13947.563147,2463.375776,2463.375776
std,7240.635548,7240.635548,2797.729421,2797.729421
min,719.000000,719.000000,3.000000,3.000000
25%,8397.500000,8397.500000,447.000000,447.000000
50%,12064.000000,12064.000000,1425.500000,1425.500000
75%,18588.500000,18588.500000,3381.500000,3381.500000
max,41128.000000,41128.000000,17627.000000,17627.000000



Products with fewer than 100 sales rank points:
0

Products with fewer than 100 offer count points:
58


,asin,sales_rank_points,sales_rank_non_missing_points,offer_count_points,offer_count_non_missing_points
866,B08V8GXQG7,6079,6079,3,3
953,B09K736J9Y,4511,4511,3,3
905,B096F3V54K,5437,5437,5,5
603,B07T663Q9P,6574,6574,13,13
795,B08J8BRKT7,6364,6364,15,15
861,B08TTK3DK2,719,719,17,17
692,B082YM66TS,4756,4756,27,27
828,B08NT1HGZW,6644,6644,28,28
569,B07QJ1X2J4,7443,7443,29,29
855,B08T6YY226,3922,3922,33,33


In [ ]:
# Check Raw Time-Series Date Coverage

date_coverage_rows = []

fields_to_check = ["AMAZON", "NEW", "SALES", "COUNT_NEW"]

for file in raw_files:
    with open(file, "r", encoding="utf-8") as f:
        product = json.load(f)
    
    asin = product.get("asin", file.stem.replace("_raw", ""))
    data_obj = product.get("data", {})
    
    row = {"asin": asin}
    
    for field in fields_to_check:
        times = data_obj.get(f"{field}_time", [])
        
        if isinstance(times, list) and len(times) > 0:
            times_dt = pd.to_datetime(times, format="%Y-%m-%d %H:%M:%S", errors="coerce")
            row[f"{field}_start"] = times_dt.min()
            row[f"{field}_end"] = times_dt.max()
            row[f"{field}_days_covered"] = (times_dt.max() - times_dt.min()).days
        else:
            row[f"{field}_start"] = pd.NaT
            row[f"{field}_end"] = pd.NaT
            row[f"{field}_days_covered"] = np.nan
    
    date_coverage_rows.append(row)

date_coverage_df = pd.DataFrame(date_coverage_rows)

coverage_summary = date_coverage_df[
    [f"{field}_days_covered" for field in fields_to_check]
].describe().T

display(
    coverage_summary.style
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

date_coverage_df.head(10).style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,count,mean,std,min,25%,50%,75%,max
AMAZON_days_covered,966.000000,2936.018634,1179.418440,0.000000,2099.250000,2748.500000,3598.250000,5547.000000
NEW_days_covered,966.000000,3008.444099,1077.457525,1588.000000,2133.250000,2780.000000,3607.500000,5547.000000
SALES_days_covered,966.000000,2886.464803,834.651979,1580.000000,2140.000000,2791.000000,3612.750000,4138.000000
COUNT_NEW_days_covered,966.000000,2832.583851,831.489309,89.000000,2106.250000,2755.500000,3609.250000,4045.000000


,asin,AMAZON_start,AMAZON_end,AMAZON_days_covered,NEW_start,NEW_end,NEW_days_covered,SALES_start,SALES_end,SALES_days_covered,COUNT_NEW_start,COUNT_NEW_end,COUNT_NEW_days_covered
0,B00002N6J9,2021-03-12 10:08:00,2026-05-07 21:22:00,1882,2021-03-12 10:08:00,2026-05-07 21:22:00,1882,2021-03-12 17:24:00,2026-05-31 23:36:00,1906,2021-03-12 10:08:00,2026-05-30 02:22:00,1904
1,B00002N86A,2014-03-18 10:00:00,2026-05-24 08:08:00,4449,2014-03-18 10:00:00,2026-06-01 05:36:00,4457,2015-02-01 22:00:00,2026-06-01 00:36:00,4137,2015-05-05 22:27:00,2026-05-30 17:22:00,4042
2,B00004RB5H,2011-04-09 04:00:00,2026-05-27 07:08:00,5527,2011-04-08 16:00:00,2026-05-27 07:08:00,5527,2015-02-03 13:00:00,2026-06-01 00:36:00,4135,2015-05-05 15:28:00,2026-05-21 00:08:00,4033
3,B00004SY4H,2014-06-09 09:00:00,2026-06-01 07:04:00,4374,2014-06-09 09:00:00,2026-06-01 07:04:00,4374,2015-02-01 16:00:00,2026-05-31 22:40:00,4137,2015-05-04 23:05:00,2026-05-29 14:36:00,4042
4,B0000510R4,2011-03-24 16:00:00,2026-05-30 17:46:00,5546,2011-03-24 16:00:00,2026-05-30 18:04:00,5546,2015-02-02 01:00:00,2026-06-01 10:40:00,4137,2015-05-09 14:00:00,2026-05-30 05:56:00,4038
5,B0000511U7,2011-03-24 16:00:00,2026-05-27 03:40:00,5542,2011-03-24 16:00:00,2026-05-27 03:40:00,5542,2015-02-02 01:00:00,2026-06-01 09:16:00,4137,2015-05-06 04:52:00,2026-05-31 11:20:00,4043
6,B000051ZHS,2015-01-23 23:00:00,2026-06-01 00:12:00,4146,2015-01-23 23:00:00,2026-05-30 02:32:00,4144,2015-02-02 00:00:00,2026-06-01 00:12:00,4137,2015-05-04 22:25:00,2026-06-01 00:12:00,4045
7,B000058AKA,2011-03-24 16:00:00,2026-05-29 03:04:00,5544,2011-03-24 16:00:00,2026-05-29 03:04:00,5544,2015-02-04 01:00:00,2026-06-01 08:16:00,4135,2015-05-09 16:30:00,2026-04-28 22:08:00,4007
8,B000058TJ3,2015-06-10 00:42:00,2026-06-01 00:04:00,4008,2015-06-10 00:42:00,2026-06-01 03:46:00,4009,2015-06-10 00:42:00,2026-06-01 09:52:00,4009,2015-06-10 00:42:00,2026-06-01 03:46:00,4009
9,B00005NKAA,2017-07-06 21:12:00,2026-05-29 01:24:00,3248,2017-07-06 21:12:00,2026-06-01 12:22:00,3251,2017-07-06 21:12:00,2026-06-01 15:28:00,3251,2017-07-06 21:12:00,2026-05-30 14:36:00,3249


# Raw Data Exploration Summary

The V2 raw dataset contains 966 saved Keepa JSON files, one per ASIN.

The raw files contain rich product-level and time-series information. The most important fields identified for the modelling pipeline are:

- `AMAZON`: Amazon direct price history
- `NEW`: new offer price history
- `SALES`: sales rank history
- `COUNT_NEW`: new offer count history
- `LISTPRICE`: list price history
- `monthlySold`: current monthly sales estimate
- `monthlySoldHistory`: historical monthly sales estimate
- `availabilityAmazon`: Amazon availability status
- `categoryTree`: product category hierarchy
- `brand` and `manufacturer`: product metadata
- `trackingSince`, `listedSince`, and `lastPriceChange`: product age and update metadata

The raw data is suitable for feature engineering because most products have substantial historical coverage. Median best available price history is around 900 non-missing price points per product, and sales rank history is especially strong, with no product having fewer than 100 sales-rank observations.

Several data quality issues were identified:

1. Some raw `NEW` prices contain unrealistic outliers, including values below $1 and extremely high values.
2. `AMAZON` price outliers are rare, but still present.
3. `COUNT_NEW` contains `-1` values, which should be treated as missing or unavailable offer-count records.
4. `LISTPRICE`, `USED`, `COUNT_USED`, and `COUNT_REFURBISHED` are noisier and less central to the project.
5. Leaf categories are highly sparse, with many categories containing only one product. Root category is therefore more suitable as a stable model feature.
6. Rating and review count are not consistently available across the raw V2 files and should not be used as core features.
7. `availabilityAmazon` requires careful handling because products not directly available from Amazon may depend more on `NEW` marketplace prices. The final dataset should retain a `price_source` feature to make this distinction explicit.

Based on this exploration, the processing script should extract the core time-series fields, apply price validity checks, preserve useful metadata, handle availability and offer-count missingness carefully, and create a clean daily dataset for feature engineering and XGBoost modelling.

## Interpretation of Time-Series Coverage

The raw V2 dataset has strong historical coverage across the main time-series fields. Median coverage is approximately 2,700 to 2,800 days for `AMAZON`, `NEW`, `SALES`, and `COUNT_NEW`, which is roughly 7.5 years of historical data for the typical product.

This is suitable for a 14-day price-drop prediction task because the lookahead window is small compared with the amount of historical data available. It also supports a chronological validation strategy, where the final months of data can be held out for testing while still leaving several years of previous observations for training.

The main limitation is that `COUNT_NEW` has weaker coverage for some products, so offer-count features should be used carefully and missingness should be handled explicitly.

## Interpretation of Amazon Availability Status

The `availabilityAmazon` field is available for all 966 raw products, but it should not be treated as a simple price field.

Most products have `availabilityAmazon = 0`, while a sizeable subset has `availabilityAmazon = -1` or other status codes. This matters because products not directly available from Amazon may rely more heavily on `NEW` offer prices rather than `AMAZON` prices.

This has implications for feature engineering and label construction. A future price drop based on `AMAZON` price may not mean the same thing as a future price drop based on third-party `NEW` prices. Therefore, the final processed dataset should retain a `price_source` feature so the model can distinguish whether the selected price came from Amazon direct prices or marketplace new-offer prices.